### **Installations & Imports**

##### **Library & Package Imports**

In [ ]:
import os
import time
import json
import csv

import matplotlib.pyplot as plt
import pandas as pd

import requests

!pip install ip2geotools
import socket
import requests
from ip2geotools.databases.noncommercial import DbIpCity
from geopy.distance import distance

import logging
logging.getLogger("geocoder").setLevel(logging.CRITICAL)

!pip install geoip2fast
from geoip2fast import GeoIP2Fast

!pip install plotly
import plotly.express as px

### **File Reading & Writing**

##### **Reading (edit data) from JSON files**

In [ ]:
def read_json_file (file_path : str):
    """
    This function returns the contents of a JSON file.

    :file_path: The file path of the JSON file to read from.

    Returns the JSON data as a list of lists, or None if the file could not be opened.
    """
    data = None
    if os.path.isfile(file_path) == False or len(file_path) <= 5 or file_path[-5:] != ".json":
        print(f"'{file_path}' is either not a JSON file, or does not exist.")
    else:
        try:
            with open(file_path, 'r', encoding="utf-8") as file:
                data = json.load(file)
                for d in data:
                    d[0] = str(d[0])
        except Exception as e:
            print(f"The file at '{file_path}' could not be read. Ensure that it contains a nested JSON list.")
            data = None

    return data

##### **Reading (edit data) from CSV files**

In [ ]:
def read_csv_file (file_path : str):
    """
    This function returns the contents of a CSV file.

    :file_path: The file path of the CSV file to read from.

    Returns the CSV data as a list of lists, or None if the file could not be opened.
    """
    data = None
    if os.path.isfile(file_path) == False or len(file_path) <= 4 or file_path[-4:] != ".csv":
        print(f"'{file_path}' is either not a CSV file, or does not exist.")
    else:
        try:
            with open(file_path, 'r', encoding="utf-8") as file:
                data = list(csv.reader(file))[1:]
                for d in data:
                    d[2] = int(d[2])
        except Exception as e:
            print(f"The file at '{file_path}' could not be read. Ensure that it contains valid CSV.")
            data = None

    return data

##### **Reading the (edit data analysis) results from JSON files**

In [ ]:
def read_results_file (file_path : str):
    """
    This function returns the analysis results stored in a JSON file.

    :file_path: The file path of the JSON file to read from.

    Returns the JSON data as a dictionary, or None if the file could not be opened.
    """
    data = None
    if os.path.isfile(file_path) == False or len(file_path) <= 5 or file_path[-5:] != ".json":
        print(f"'{file_path}' is either not a JSON file, or does not exist.")
    else:
        try:
            with open(file_path, 'r', encoding="utf-8") as file:
                data = json.load(file)
        except Exception as e:
            print(f"The file at '{file_path}' could not be read. Ensure that it contains valid JSON.")
            data = None

    return data

##### **Writing the (edit data analysis) results to a JSON file**

In [ ]:
def save_results (file_path : str, results_data : dict, results_name : str):
    """
    This function saves the output of the full_edit_data_analysis() function to a JSON file.

    :file_path: The file path of the JSON file to write to. If no such file exists, it will be created automatically.
    :results_data: A dictionary containing the results data outputted from the full_edit_data_analysis() function.
    :results_name: A string denoting the name (dictionary key) that the results data should be stored under.

    Returns True if the results data was saved successfully, and False otherwise.
    """
    if len(file_path) <= 5 or file_path[-5:] != ".json":
        print(f"'{file_path}' is not the file path for a JSON file.")
        return False

    file_data = read_results_file(file_path)
    if file_data == None:
        file_data = {}

    if type(results_data) != dict or type(results_name) != str:
        print("Error: results_data must be a dictionary, and results_name must be a string.")
        return False

    file_data[results_name] = results_data

    with open(file_path, "w") as file:
        json.dump(file_data, file, indent=4, ensure_ascii=False)

    return True

##### **Dictionary of ISO 3166-1 country codes**

In [ ]:
# Dictionary source: https://stackoverflow.com/a/16263580

iso_country_codes = {'AF': 'Afghanistan', 'AL': 'Albania', 'DZ': 'Algeria', 'AS': 'American Samoa', 'AD': 'Andorra', 'AO': 'Angola', 'AI': 'Anguilla', 'AQ': 'Antarctica', 'AG': 'Antigua and Barbuda', 'AR': 'Argentina', 'AM': 'Armenia', 'AW': 'Aruba', 'AU': 'Australia', 'AT': 'Austria', 'AZ': 'Azerbaijan', 'BS': 'Bahamas', 'BH': 'Bahrain', 'BD': 'Bangladesh', 'BB': 'Barbados', 'BY': 'Belarus', 'BE': 'Belgium', 'BZ': 'Belize', 'BJ': 'Benin', 'BM': 'Bermuda', 'BT': 'Bhutan', 'BO': 'Bolivia, Plurinational State of', 'BQ': 'Bonaire, Sint Eustatius and Saba', 'BA': 'Bosnia and Herzegovina', 'BW': 'Botswana', 'BV': 'Bouvet Island', 'BR': 'Brazil', 'IO': 'British Indian Ocean Territory', 'BN': 'Brunei Darussalam', 'BG': 'Bulgaria', 'BF': 'Burkina Faso', 'BI': 'Burundi', 'KH': 'Cambodia', 'CM': 'Cameroon', 'CA': 'Canada', 'CV': 'Cape Verde', 'KY': 'Cayman Islands', 'CF': 'Central African Republic', 'TD': 'Chad', 'CL': 'Chile', 'CN': 'China', 'CX': 'Christmas Island', 'CC': 'Cocos (Keeling) Islands', 'CO': 'Colombia', 'KM': 'Comoros', 'CG': 'Congo', 'CD': 'Congo, the Democratic Republic of the', 'CK': 'Cook Islands', 'CR': 'Costa Rica', 'HR': 'Croatia', 'CU': 'Cuba', 'CW': 'Curaçao', 'CY': 'Cyprus', 'CZ': 'Czech Republic', 'CI': "Côte d'Ivoire", 'DK': 'Denmark', 'DJ': 'Djibouti', 'DM': 'Dominica', 'DO': 'Dominican Republic', 'EC': 'Ecuador', 'EG': 'Egypt', 'SV': 'El Salvador', 'GQ': 'Equatorial Guinea', 'ER': 'Eritrea', 'EE': 'Estonia', 'ET': 'Ethiopia', 'FK': 'Falkland Islands (Malvinas)', 'FO': 'Faroe Islands', 'FJ': 'Fiji', 'FI': 'Finland', 'FR': 'France', 'GF': 'French Guiana', 'PF': 'French Polynesia', 'TF': 'French Southern Territories', 'GA': 'Gabon', 'GM': 'Gambia', 'GE': 'Georgia', 'DE': 'Germany', 'GH': 'Ghana', 'GI': 'Gibraltar', 'GR': 'Greece', 'GL': 'Greenland', 'GD': 'Grenada', 'GP': 'Guadeloupe', 'GU': 'Guam', 'GT': 'Guatemala', 'GG': 'Guernsey', 'GN': 'Guinea', 'GW': 'Guinea-Bissau', 'GY': 'Guyana', 'HT': 'Haiti', 'HM': 'Heard Island and McDonald Islands', 'VA': 'Holy See (Vatican City State)', 'HN': 'Honduras', 'HK': 'Hong Kong', 'HU': 'Hungary', 'IS': 'Iceland', 'IN': 'India', 'ID': 'Indonesia', 'IR': 'Iran, Islamic Republic of', 'IQ': 'Iraq', 'IE': 'Ireland', 'IM': 'Isle of Man', 'IL': 'Israel', 'IT': 'Italy', 'JM': 'Jamaica', 'JP': 'Japan', 'JE': 'Jersey', 'JO': 'Jordan', 'KZ': 'Kazakhstan', 'KE': 'Kenya', 'KI': 'Kiribati', 'KP': "Korea, Democratic People's Republic of", 'KR': 'Korea, Republic of', 'KW': 'Kuwait', 'KG': 'Kyrgyzstan', 'LA': "Lao People's Democratic Republic", 'LV': 'Latvia', 'LB': 'Lebanon', 'LS': 'Lesotho', 'LR': 'Liberia', 'LY': 'Libya', 'LI': 'Liechtenstein', 'LT': 'Lithuania', 'LU': 'Luxembourg', 'MO': 'Macao', 'MK': 'Macedonia, the former Yugoslav Republic of', 'MG': 'Madagascar', 'MW': 'Malawi', 'MY': 'Malaysia', 'MV': 'Maldives', 'ML': 'Mali', 'MT': 'Malta', 'MH': 'Marshall Islands', 'MQ': 'Martinique', 'MR': 'Mauritania', 'MU': 'Mauritius', 'YT': 'Mayotte', 'MX': 'Mexico', 'FM': 'Micronesia, Federated States of', 'MD': 'Moldova, Republic of', 'MC': 'Monaco', 'MN': 'Mongolia', 'ME': 'Montenegro', 'MS': 'Montserrat', 'MA': 'Morocco', 'MZ': 'Mozambique', 'MM': 'Myanmar', 'NA': 'Namibia', 'NR': 'Nauru', 'NP': 'Nepal', 'NL': 'Netherlands', 'NC': 'New Caledonia', 'NZ': 'New Zealand', 'NI': 'Nicaragua', 'NE': 'Niger', 'NG': 'Nigeria', 'NU': 'Niue', 'NF': 'Norfolk Island', 'MP': 'Northern Mariana Islands', 'NO': 'Norway', 'OM': 'Oman', 'PK': 'Pakistan', 'PW': 'Palau', 'PS': 'Palestine, State of', 'PA': 'Panama', 'PG': 'Papua New Guinea', 'PY': 'Paraguay', 'PE': 'Peru', 'PH': 'Philippines', 'PN': 'Pitcairn', 'PL': 'Poland', 'PT': 'Portugal', 'PR': 'Puerto Rico', 'QA': 'Qatar', 'RO': 'Romania', 'RU': 'Russian Federation', 'RW': 'Rwanda', 'RE': 'Réunion', 'BL': 'Saint Barthélemy', 'SH': 'Saint Helena, Ascension and Tristan da Cunha', 'KN': 'Saint Kitts and Nevis', 'LC': 'Saint Lucia', 'MF': 'Saint Martin (French part)', 'PM': 'Saint Pierre and Miquelon', 'VC': 'Saint Vincent and the Grenadines', 'WS': 'Samoa', 'SM': 'San Marino', 'ST': 'Sao Tome and Principe', 'SA': 'Saudi Arabia', 'SN': 'Senegal', 'RS': 'Serbia', 'SC': 'Seychelles', 'SL': 'Sierra Leone', 'SG': 'Singapore', 'SX': 'Sint Maarten (Dutch part)', 'SK': 'Slovakia', 'SI': 'Slovenia', 'SB': 'Solomon Islands', 'SO': 'Somalia', 'ZA': 'South Africa', 'GS': 'South Georgia and the South Sandwich Islands', 'SS': 'South Sudan', 'ES': 'Spain', 'LK': 'Sri Lanka', 'SD': 'Sudan', 'SR': 'Suriname', 'SJ': 'Svalbard and Jan Mayen', 'SZ': 'Swaziland', 'SE': 'Sweden', 'CH': 'Switzerland', 'SY': 'Syrian Arab Republic', 'TW': 'Taiwan, Province of China', 'TJ': 'Tajikistan', 'TZ': 'Tanzania, United Republic of', 'TH': 'Thailand', 'TL': 'Timor-Leste', 'TG': 'Togo', 'TK': 'Tokelau', 'TO': 'Tonga', 'TT': 'Trinidad and Tobago', 'TN': 'Tunisia', 'TR': 'Turkey', 'TM': 'Turkmenistan', 'TC': 'Turks and Caicos Islands', 'TV': 'Tuvalu', 'UG': 'Uganda', 'UA': 'Ukraine', 'AE': 'United Arab Emirates', 'GB': 'United Kingdom', 'US': 'United States', 'UM': 'United States Minor Outlying Islands', 'UY': 'Uruguay', 'UZ': 'Uzbekistan', 'VU': 'Vanuatu', 'VE': 'Venezuela, Bolivarian Republic of', 'VN': 'Viet Nam', 'VG': 'Virgin Islands, British', 'VI': 'Virgin Islands, U.S.', 'WF': 'Wallis and Futuna', 'EH': 'Western Sahara', 'YE': 'Yemen', 'ZM': 'Zambia', 'ZW': 'Zimbabwe', 'AX': 'Åland Islands'}

### **Analysis Functions**

##### **Identifying a country from an IP address**

In [ ]:
geo = GeoIP2Fast()

def ip_to_country(ip : str):
    """
    This function takes an IP address (v4 or v6) and returns the country associated with it.

    :ip: A string containing the IP address to convert.

    Returns the country name as a string.
    """
    v6 = ip[:3] == "v6-" # Detect whether the IP address is v4 or v6

    new_ip = ""
    country = ""

    # Convert the IP address into the conventional format
    if v6 == True:
        # If this is an IPv6 address
        for i in range(3, 35, 4):
            new_ip += ip[i:i+4]
            if i != 31:
                new_ip += ":"
        country_code = DbIpCity.get(new_ip, api_key="free").country
        if country_code not in iso_country_codes:
            country = ""
        else:
            country = iso_country_codes[country_code]

    else:
        # If this is an IPv4 address
        for i in range(0, 8, 2):
            new_ip += str(int(ip[i:i+2], 16))
            if i != 6:
                new_ip += "."
        country_code =  geo.lookup(new_ip).country_code
        if country_code not in iso_country_codes:
            country = ""
        else:
            country = iso_country_codes[country_code]

    return country

##### **Couting the number of edits from each year**

In [ ]:
def edits_per_year(edit_data : list, proportion : bool = False):
    """
    This function takes a list of edit data, and counts the number of edits made in each year.

    :edit_data: A list containing the Wikipedia edit data, in the format [timestamp, IP address, editor ID].
    :proportion: Set to True/False to return the proportion/number of edits per year. Default is False.

    Returns a dictionary in the format {year : number/proportion of edits}.
    """
    if type(edit_data) != list or len(edit_data) == 0:
        return {}

    year_dict = dict()
    for edit in edit_data:
        if edit[0][0:4] in year_dict:
            year_dict[edit[0][0:4]] += 1
        else:
            year_dict[edit[0][0:4]] = 1

    if proportion == True:
        for year in year_dict:
            year_dict[year] = round(year_dict[year] / len(edit_data), 5)

    return year_dict

##### **Counting the number of edits and editors from each country**

In [ ]:
def edits_editors_per_country(edit_data : list, proportion : bool = False):
    """
    This function takes a list of edit data, and counts the number of edits and editors from each country.

    :edit_data: A list containing the Wikipedia edit data, in the format [timestamp, IP address, editor ID].
    :proportion: Set to True/False to return the proportion/number of edits and editors per country. Default is False.

    Returns two dictionaries: {country : number/proportion of edits}, {country : number/proportion of editors}.
    """
    if type(edit_data) != list or len(edit_data) == 0:
        return {}

    country_edits = dict()
    country_editors = dict()
    editor_ids = list()
    num_editors = 0

    for edit in edit_data:
        country = None
        while country == None:
            try:
                country = ip_to_country(edit[1])
            except:
                print("Ip2geotools request limit reached - waiting 60 seconds before trying again.")
                time.sleep(60)

        if country != "":
            if country in country_edits:
                country_edits[country] += 1
            else:
                country_edits[country] = 1

            if country in country_editors and edit[2] not in editor_ids:
                editor_ids.append(edit[2])
                country_editors[country] += 1
                num_editors += 1
            elif country not in country_editors and edit[2] not in editor_ids:
                editor_ids.append(edit[2])
                country_editors[country] = 1
                num_editors += 1

    if proportion == True:
        for country in country_edits:
            country_edits[country] = round(country_edits[country] / len(edit_data), 5)
        for country in country_editors:
            country_editors[country] = round(country_editors[country] / num_editors, 5)

    # Sort the dictionaries
    country_edits = dict(sorted(country_edits.items(), key=lambda item: item[1], reverse=True))
    country_editors = dict(sorted(country_editors.items(), key=lambda item: item[1], reverse=True))

    return country_edits, country_editors

### **Results Generation**

##### **Master edit data analysis function**

In [ ]:
def full_edit_data_analysis(edit_data_file : str, proportion : bool = False):
    """
    This function performs the full edit data analysis process for a given article edition.

    :edit_data_file: The name/path of the text file containing the edit data.
    :proportion: Set to True/False to return the proportion/number of edits and editors per country. Default is False.

    Returns a dictionary with the following values in relation to the given article edition:
    'total_edits' (int) - The total number of edits (from anonymous editors) received.
    'edits_year' (dict) - The number of edits received in each year.
    'edits_country' (dict) - The number of edits received from each country.
    'editors_country' (dict) - The number of editors from each country.
    """

    # Read the edit data file
    edit_data = list()
    if len(edit_data_file) > 5 and edit_data_file[-5:] == '.json':
        edit_data = read_json_file(edit_data_file)
    elif len(edit_data_file) > 4 and edit_data_file[-4:] == '.csv':
        edit_data = read_csv_file(edit_data_file)
    else:
        print(f"The file at '{edit_data_file}' must be a JSON or CSV file.")
        return

    # Get the total number of edits
    total_edits = len(edit_data)

    # Get the number of edits from each year
    edits_year = edits_per_year(edit_data, proportion)

    # Get the number of edits and editors from each country
    edits_country, editors_country = edits_editors_per_country(edit_data, proportion)

    # Return the results
    results_dict = {
        "total_edits" : total_edits,
        "edits_year" : edits_year,
        "edits_country" : edits_country,
        "editors_country" : editors_country
    }

    return results_dict

### **Results Visualisation**

##### **Visualising the number of edits from each year (using a line chart)**

In [ ]:
def years_line_chart(results : dict, title_prefix : str):
    """
    This function generates a line chart of the proportion of edits recieved in each year by one or more Wikipedia article editions.

    :results: A dictionary containing the edit data analysis results for the article.
    This is expected to be the ouput of the full_edit_data_analysis() function.
    :title_prefix: A string containing the prefix of the title to be displayed above the line chart.
    """
    rl = len(results)
    if type(results) != dict or rl == 0:
        print("The given results are invalid.")
        return

    # The list of colours used for the plotted points of different article editions.
    color_list = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'olive', 'pink', 'gray', 'brown']

    earliest = [2000, False]
    latest = [2027, False]

    while earliest[1] == False and earliest[0] < 2026:
        earliest[0] += 1
        for edition in results:
            if str(earliest[0]) in results[edition]["edits_year"]:
                earliest[1] = True
                break

    while latest[1] == False and latest[0] > 2001:
        latest[0] -= 1
        for edition in results:
            if str(latest[0]) in results[edition]["edits_year"]:
                latest[1] = True
                break

    plt.figure(figsize=(12, 6))

    for e in range(rl):
        years = []
        percents = []
        for y in range(earliest[0], latest[0] + 1):
            years.append(str(y))
            if str(y) in results[list(results)[e]]["edits_year"]:
                percents.append(results[list(results)[e]]["edits_year"][str(y)] * 100)
            else:
                percents.append(0)
        plt.plot(years, percents,  marker='o', linestyle='-', label=list(results)[e], color=color_list[e], alpha=0.5)

    plt.grid(which='major', linewidth=0.8, alpha=0.9)
    plt.grid(which='minor', linestyle=':', linewidth=0.6, axis = 'y', alpha=0.7)
    plt.minorticks_on()
    plt.xlabel("Year", fontsize=11, fontweight=600)
    plt.ylabel(f"% of edits", fontsize=11, fontweight=600)
    plt.xticks(rotation=45)
    plt.title(f"{title_prefix} - % of Edits Per Year", fontsize=14, fontweight=600)
    plt.legend()
    plt.show()

##### **Visualising the global distribution of edits and editors (using bar charts)**

In [ ]:
def countries_bar_charts(results : dict, title_prefix : str):
    """
    This function generates a bar chart of the proportion of edits made by, and editors belonging to,
    the top 10 highest countributors (countries) for a Wikipedia article edition.

    :results: A dictionary containing the edit data analysis results for the article.
    This is expected to be the ouput of the full_edit_data_analysis() function.
    :title_prefix: A string containing the prefix of the title to be displayed above the line chart.
    """
    rl = len(results)
    if type(results) != dict or rl == 0:
        print("The given results are invalid.")
        return

    # The list of colours used for the plotted points of different article editions.
    color_list = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'olive', 'pink', 'gray', 'brown']

    for e in range(rl):
        plt.figure(figsize=(10, 5))
        edits_country = results[list(results)[e]]["edits_country"]
        editors_country = results[list(results)[e]]["editors_country"]

        countries = 10
        if len(edits_country) < 10:
            countries =  len(edits_country)
        x = range(countries)

        y1 = [(edits_country[list(edits_country)[c]] * 100) for c in x] # % edits from top 10 countries
        y2 = [(editors_country[list(editors_country)[c]] * 100) for c in x] # % editors from top 10 countries
        #y3 = [0.1 for c in x] # % editors for the entire Wikipedia edition from top 10 countries

        plt.bar([n-0.2 for n in x], y1, 0.4, color=color_list[0], edgecolor='black', alpha=0.75)
        plt.bar([n+0.2 for n in x], y2, 0.4, color=color_list[1], edgecolor='black', alpha=0.75)
        #plt.bar([n+0.3 for n in x], y3, 0.3, color=color_list[2], edgecolor='black', alpha=0.75)

        country_names = []
        for country in list(edits_country)[0:countries]:
            if ',' in country:
                country_names.append(country.split(',')[0])
            else:
                country_names.append(country)

        plt.xticks(x, country_names)
        plt.xticks(rotation=45)
        plt.xlabel("Country", fontsize=11, fontweight=600)
        plt.ylabel(f"% of edits/editors", fontsize=11, fontweight=600)
        plt.legend(["Edits (this article)", "Editors (this article)"]) # , f"Editors (entire {list(results)[e]} edition)"
        plt.title(f"{title_prefix} - % of Edits & Editors Per Country ({list(results)[e]})", fontsize=14, fontweight=600)
        plt.gca().set_axisbelow(True)
        plt.grid(axis = 'y')
        plt.grid(which='minor', linestyle=':', linewidth=0.6, axis = 'y', alpha=0.7)
        plt.minorticks_on()
        plt.show()
        print()


##### **Visualising the global distribution of edits and editors (using choropleth maps)**

In [ ]:
def countries_choropleth_maps(results : dict, title_prefix : str, editors : bool):
    """
    This function generates a choropleth map showing the proportion of editors from every country,
    for a Wikipedia article edition.

    :results: A dictionary containing the edit data analysis results for the article.
    This is expected to be the ouput of the full_edit_data_analysis() function.
    :title_prefix: A string containing the prefix of the title to be displayed above the line chart.
    """
    rl = len(results)
    if type(results) != dict or rl == 0:
        print("The given results are invalid.")
        return

    # The list of colours used for the plotted points of different article editions.
    color_list = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'olive', 'pink', 'gray', 'brown']

    for e in range(rl):
        if editors == False:
            data = {
                "country" : list(results[list(results)[e]]["edits_country"]),
                "value": list(results[list(results)[e]]["edits_country"].values())
            }
            map_text = "% of Edits"
        else:
            data = {
                "country" : list(results[list(results)[e]]["editors_country"]),
                "value": list(results[list(results)[e]]["editors_country"].values())
            }
            map_text = "% of Editors"

        data["value"] = [v*100 for v in data["value"]]

        bins = [0, 0.5, 1, 3, 5, 10, 25, 50, 100]
        bin_labels = [f"{bins[i]}-{bins[i+1]}%" for i in range(len(bins)-1)]
        bin_colours = {
            "0-0.5%" : "#d2d9fd", "0.5-1%" : "#aec0f8", "1-3%" : "#8ba7f3", "3-5%" : "#688eee",
            "5-10%" : "#4e72c9", "10-25%" : "#34569d", "25-50%" : "#1a3a71", "50-100%" : "#011f45"
        }
        data[map_text] = pd.cut(data["value"], bins=bins, labels=bin_labels, include_lowest=True)

        fig = px.choropleth(
            data_frame = pd.DataFrame(data),
            locations = "country",
            color = map_text,
            color_discrete_map = bin_colours, # Comment out this line to use more colourful colours
            hover_name = "country",
            locationmode = "country names",
            title = f"{title_prefix} - {map_text} Per Country ({list(results)[e]})"
        )
        fig.update_layout(
            width=1000,
            height=500,
            margin=dict(l=5, r=5, t=30, b=5),
            title={'x' : 0.5, 'xanchor' : 'center', 'yanchor' : 'top',
                    'font' : {'size': 20, 'color': "black", 'family': "Arial Black"}}
        )
        fig.show()
        print()

### **Code Execution Area**

##### **Running the master function**

In [ ]:
results_1 = full_edit_data_analysis("Edit_Data_English.csv", True)
print("English results:")
for key, value in results_1.items():
    print(f"{key} : {value}")

results_2 = full_edit_data_analysis("Edit_Data_Russian.json", True)
print("\nRussian results:")
for key, value in results_2.items():
    print(f"{key} : {value}")

results_3 = full_edit_data_analysis("Edit_Data_Spanish.json", True)
print("\nSpanish results:")
for key, value in results_3.items():
    print(f"{key} : {value}")

results_4 = full_edit_data_analysis("Edit_Data_Vietnamese.json", True)
print("\nVietnamese results:")
for key, value in results_4.items():
    print(f"{key} : {value}")

##### **Saving the results (as JSON files)**

In [ ]:
save_results("edit_data_analysis_results.json", results_1, "English")
save_results("edit_data_analysis_results.json", results_2, "Russian")
save_results("edit_data_analysis_results.json", results_3, "Spanish")
save_results("edit_data_analysis_results.json", results_4, "Vietnamese")

##### **Generating visualisations of the number of edits each year (line chart)**

In [ ]:
results = read_results_file("edit_data_analysis_results.json")

years_line_chart(results, "John Kennedy")

##### **Generating visualisations of the global distribution of edits and editors (bar chart)**

In [ ]:
results = read_results_file("edit_data_analysis_results.json")

countries_bar_charts(results, "John Kennedy")

##### **Generating visualisations of the global distribution of editors only (choropleth map)**


In [ ]:
results = read_results_file("edit_data_analysis_results.json")

countries_choropleth_maps(results, "John Kennedy", True)